In [3]:
from langchain_groq import ChatGroq

In [5]:
llm = ChatGroq(model='openai/gpt-oss-120b')

res1 = llm.invoke('Hii.. I am Manish')

res2 = llm.invoke('Do u rememner my name')

print(res1.content)
print(res2.content)

Hello Manish! 👋 Nice to meet you. How can I assist you today?
I don’t have a record of your name from our previous messages. If you’d like to share it again, I can use it while we’re chatting, and I’ll keep it private. Let me know how you’d like to be addressed!


From u can see that model dont have any memory so in order to give ot memory we need to give it memory using some packages

In [14]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

In [10]:
msg_hst = ChatMessageHistory()

In [11]:
# msg_hst.add_user_message('Hello')

# msg_hst.add_ai_message('Hii How can i help you today')

# print(msg_hst)

# type(msg_hst)

Human: Hello
AI: Hii How can i help you today


langchain_core.chat_history.InMemoryChatMessageHistory

In [32]:
store = {}

In [33]:
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [34]:
llm = ChatGroq(model='openai/gpt-oss-120b')

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
	('system', 'You are helpful AI Assistant'),
	('human', '{input}')
])

In [35]:
chain = prompt | llm | parser

In [36]:
chat_with_history = RunnableWithMessageHistory(
    chain,
    get_history, 
    input_messages_key="input", 
    output_messages_key="output"
)

In [37]:
response1 = chat_with_history.invoke(
    {
        "input": "Hello I am Manish",
    },
    config={"configurable": {"session_id": "manish123"}}
)
response1

'Hello Manish! How can I help you today?'

In [38]:
get_history('manish123')

InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello I am Manish', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Manish! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [39]:
response2 = chat_with_history.invoke(
    {
        "input": "Do u remember my name",
    },
    config={"configurable": {"session_id": "manish123"}}
)
response2

'Yes, I remember—your name is Manish. How can I assist you today?'

In [40]:
store

{'manish123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello I am Manish', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Manish! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Do u remember my name', additional_kwargs={}, response_metadata={}), AIMessage(content='Yes, I remember—your name is Manish. How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}